In [1]:
from sklearn.linear_model import LogisticRegression
import numpy as np
import pandas as pd

In [2]:
x_train = pd.read_csv('x_train_scaled.csv')
x_test = pd.read_csv('x_test_scaled.csv')
y_train = pd.read_csv('y_train.csv')
y_test = pd.read_csv('y_test.csv')

x_train.head()

,RevolvingUtilizationOfUnsecuredLines,age,NumberOfTime30-59DaysPastDueNotWorse,DebtRatio,MonthlyIncome,NumberOfOpenCreditLinesAndLoans,NumberOfTimes90DaysLate,NumberRealEstateLoansOrLines,NumberOfTime60-89DaysPastDueNotWorse,NumberOfDependents
0,-0.022803,-1.576975,-0.100746,-0.180840,-0.151997,-0.672522,-0.063764,-0.900750,-0.057706,-0.666180
1,-0.020393,0.184209,-0.100746,-0.180416,-0.334990,0.493412,-0.063764,-0.017611,-0.057706,-0.666180
2,-0.022803,-0.628645,-0.100746,-0.180824,-0.164834,-1.255488,-0.063764,-0.900750,-0.057706,2.944145
3,-0.021217,-0.831859,-0.100746,-0.180815,0.187732,-1.255488,-0.063764,-0.900750,-0.057706,3.846727
4,-0.022803,-1.170548,-0.100746,-0.180777,-0.045659,0.687734,-0.063764,-0.017611,-0.057706,-0.666180


In [ ]:
## Train a baseline logistic regression model_1
model_1 = LogisticRegression(
    penalty="l2",
    C=1.0,
    max_iter=1000,
    solver="lbfgs",
    n_jobs=-1,
    class_weight="balanced"
)

model_1.fit(x_train, y_train)

C:\Users\Saad\AppData\Roaming\Python\Python313\site-packages\sklearn\utils\validation.py:1406: DataConversionWarning: A column-vector y was passed when a 1d array was expected. Please change the shape of y to (n_samples, ), for example using ravel().
  y = column_or_1d(y, warn=True)


,penalty,'l2'
,dual,False
,tol,0.0001
,C,1.0
,fit_intercept,True
,intercept_scaling,1
,class_weight,'balanced'
,random_state,None
,solver,'lbfgs'
,max_iter,1000
,multi_class,'deprecated'


In [ ]:
from sklearn.metrics import roc_auc_score, roc_curve, classification_report, confusion_matrix

# predicted probabilities for the positive class (1)
y_proba = model_1.predict_proba(x_test)[:, 1]
y_pred = (y_proba >= 0.5).astype(int)  # can tune this threshold

auc = roc_auc_score(y_test, y_proba)
cm = confusion_matrix(y_test, y_pred)

print("AUC-ROC:", auc)
print("Confusion matrix:\n", cm)
print("\nClassification report:\n", classification_report(y_test, y_pred))


AUC-ROC: 0.7899659904870495
Confusion matrix:
 [[21949  6095]
 [  688  1268]]

Classification report:
               precision    recall  f1-score   support

           0       0.97      0.78      0.87     28044
           1       0.17      0.65      0.27      1956

    accuracy                           0.77     30000
   macro avg       0.57      0.72      0.57     30000
weighted avg       0.92      0.77      0.83     30000



In [ ]:
test_data = pd.read_csv('cs-test.csv')
test_ids = test_data['Unnamed: 0']
test_data_processed = pd.read_csv('processed_test_data.csv')

test_proba = model_1.predict_proba(test_data_processed)[:, 1]

test_data_processed.head()



,RevolvingUtilizationOfUnsecuredLines,age,NumberOfTime30-59DaysPastDueNotWorse,DebtRatio,MonthlyIncome,NumberOfOpenCreditLinesAndLoans,NumberOfTimes90DaysLate,NumberRealEstateLoansOrLines,NumberOfTime60-89DaysPastDueNotWorse,NumberOfDependents
0,-0.019219,-0.628645,-0.100746,-0.180755,-0.052952,-0.866844,-0.063764,-0.900750,-0.057706,-0.666180
1,-0.020928,0.319685,-0.100746,-0.180574,0.198016,1.270700,-0.063764,2.631804,-0.057706,1.138982
2,-0.022628,0.455161,-0.100746,-0.180492,-0.097953,0.687734,-0.063764,-0.017611,-0.057706,1.138982
3,-0.021669,-0.967335,0.137824,-0.180369,-0.235288,-0.283877,-0.063764,0.865527,-0.057706,-0.666180
4,-0.018755,-1.712451,-0.100746,-0.180836,-0.186787,-0.866844,-0.063764,-0.900750,-0.057706,0.236401


In [10]:
### Gradient Booster Classifier
from sklearn.ensemble import GradientBoostingClassifier

model_2 = GradientBoostingClassifier(
    n_estimators=100,
    learning_rate=0.1,
    max_depth=3,
    random_state=42
)

model_2.fit(x_train, y_train)

test_pred_proba = model_2.predict_proba(x_test)[:, 1]
threshold = 0.3

auc = roc_auc_score(y_test, test_pred_proba)
cm = confusion_matrix(y_test, (test_pred_proba >= threshold).astype(int))

print("AUC-ROC:", auc)
print("Confusion matrix:\n", cm)
print("\nClassification report:\n", classification_report(y_test, (test_pred_proba >= threshold).astype(int)))

C:\Users\Saad\AppData\Roaming\Python\Python313\site-packages\sklearn\preprocessing\_label.py:110: DataConversionWarning: A column-vector y was passed when a 1d array was expected. Please change the shape of y to (n_samples, ), for example using ravel().
  y = column_or_1d(y, warn=True)


AUC-ROC: 0.8627635228631374
Confusion matrix:
 [[27140   904]
 [ 1191   765]]

Classification report:
               precision    recall  f1-score   support

           0       0.96      0.97      0.96     28044
           1       0.46      0.39      0.42      1956

    accuracy                           0.93     30000
   macro avg       0.71      0.68      0.69     30000
weighted avg       0.93      0.93      0.93     30000



In [11]:
submission = pd.DataFrame({
    'id': test_ids,
    'probability': test_pred_proba
})


submission.to_csv('submission.csv', index=False)

ValueError: array length 30000 does not match index length 101503

In [7]:
submission.head()

,id,probability
0,1,0.409438
1,2,0.385252
2,3,0.348134
3,4,0.625096
4,5,0.557129
